In [1]:
import os
import shutil
from dotenv import load_dotenv
from langchain_community.document_loaders import DirectoryLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

load_dotenv()

print("\nLoading documents with PyMuPDF...")
loader = DirectoryLoader("data/", glob="**/*.pdf", loader_cls=PyMuPDFLoader)
docs = loader.load()
print(f"Successfully parsed {len(docs)} pages.")

# --- ORIGINAL: small chunks for precise retrieval ---
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=400, 
    chunk_overlap=80,
    add_start_index=True,
    separators=["\n\n", "\n", ".", " ", ""]
)
chunks = text_splitter.split_documents(docs)
print(f"Small chunks (400): {len(chunks)}")

# --- NEW: large chunks for context-rich retrieval ---
large_text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1200,
    chunk_overlap=300,
    add_start_index=True,
    separators=["\n\n", "\n", ".", " ", ""]
)
large_chunks = large_text_splitter.split_documents(docs)
print(f"Large chunks (1200): {len(large_chunks)}")


Loading documents with PyMuPDF...
Successfully parsed 161 pages.
Small chunks (400): 1881
Large chunks (1200): 700


In [2]:
import os
from langchain_community.vectorstores import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
import torch_directml

print("Loading BGE-Large-En-v1.5 Embedding Model on AMD GPU...")

model_kwargs = {'device': torch_directml.device()}
encode_kwargs = {
    'normalize_embeddings': True,
    'batch_size': 4
}

embedding_function = HuggingFaceEmbeddings(
    model_name="BAAI/bge-large-en-v1.5",
    model_kwargs=model_kwargs,
    encode_kwargs=encode_kwargs
)
print("Model loaded successfully!")

# --- ORIGINAL: small-chunk vectorstore ---
small_db_path = "chroma_db"
if os.path.exists(small_db_path) and os.listdir(small_db_path):
    print(f"Loading existing small-chunk DB from {small_db_path}...")
    vectorstore = Chroma(
        persist_directory=small_db_path,
        embedding_function=embedding_function
    )
else:
    print(f"Building NEW small-chunk DB at {small_db_path}...")
    vectorstore = Chroma.from_documents(
        documents=chunks,
        embedding=embedding_function,
        persist_directory=small_db_path
    )
    print("Small-chunk DB saved!")

# --- NEW: large-chunk vectorstore ---
large_db_path = "chroma_db_large"
if os.path.exists(large_db_path) and os.listdir(large_db_path):
    print(f"Loading existing large-chunk DB from {large_db_path}...")
    large_vectorstore = Chroma(
        persist_directory=large_db_path,
        embedding_function=embedding_function
    )
else:
    print(f"Building NEW large-chunk DB at {large_db_path}...")
    large_vectorstore = Chroma.from_documents(
        documents=large_chunks,
        embedding=embedding_function,
        persist_directory=large_db_path
    )
    print("Large-chunk DB saved!")

Loading BGE-Large-En-v1.5 Embedding Model on AMD GPU...
Model loaded successfully!
Building NEW small-chunk DB at chroma_db...
Small-chunk DB saved!
Building NEW large-chunk DB at chroma_db_large...
Large-chunk DB saved!


In [ ]:
import os
import yaml
from langchain_groq import ChatGroq
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers.ensemble import EnsembleRetriever
from langchain_classic.retrievers import ContextualCompressionRetriever
from langchain_community.cross_encoders import HuggingFaceCrossEncoder
from langchain_classic.retrievers.document_compressors import CrossEncoderReranker
from langchain_core.retrievers import BaseRetriever
from langchain_core.documents import Document
from langchain_core.output_parsers import StrOutputParser
from langchain_community.chat_message_histories import SQLChatMessageHistory # <-- ADDED FOR MEMORY
from typing import List

llm = ChatGroq(model="llama-3.1-8b-instant", temperature=0, max_tokens=2048)

# ==========================================
# SPAN-OVERLAP DEDUPLICATION
# ==========================================
def get_char_span(doc: Document):
    """Extract (source, start_index, end_index) from doc metadata."""
    source = doc.metadata.get('source', '')
    start  = doc.metadata.get('start_index', None)
    end    = start + len(doc.page_content) if start is not None else None
    return source, start, end

def overlap_ratio(a_start, a_end, b_start, b_end) -> float:
    """Fraction of the shorter span that overlaps with the longer span."""
    if None in (a_start, a_end, b_start, b_end):
        return 0.0
    overlap = max(0, min(a_end, b_end) - max(a_start, b_start))
    shorter = min(a_end - a_start, b_end - b_start)
    return overlap / shorter if shorter > 0 else 0.0

def deduplicate_by_span(docs: List[Document], threshold: float = 0.6) -> List[Document]:
    """Remove docs whose character span overlaps >= threshold."""
    kept = []
    seen_hashes = set()

    for doc in docs:
        src, start, end = get_char_span(doc)

        if start is not None:
            duplicate = False
            for k in kept:
                k_src, k_start, k_end = get_char_span(k)
                if k_src == src and overlap_ratio(start, end, k_start, k_end) >= threshold:
                    duplicate = True
                    break
            if not duplicate:
                kept.append(doc)
        else:
            h = hash(doc.page_content.strip())
            if h not in seen_hashes:
                seen_hashes.add(h)
                kept.append(doc)

    return kept

# ==========================================
# DUAL-RETRIEVER WRAPPER
# ==========================================
class DualEnsembleRetriever(BaseRetriever):
    small_retriever: BaseRetriever
    large_retriever: BaseRetriever
    dedup_threshold: float = 0.6

    def _get_relevant_documents(self, query: str) -> List[Document]:
        small_docs = self.small_retriever.invoke(query)
        large_docs = self.large_retriever.invoke(query)

        merged = small_docs + large_docs
        deduped = deduplicate_by_span(merged, threshold=self.dedup_threshold)

        print(f"[DualRetriever] small={len(small_docs)}, large={len(large_docs)}, "
              f"after dedup={len(deduped)}")
        return deduped

# ==========================================
# ADVANCED RAG ARCHITECTURE
# ==========================================

# -- Small-chunk retrieval leg --
vector_retriever = vectorstore.as_retriever(
    search_type='mmr', search_kwargs={"k": 20, "fetch_k": 40,"lambda_mult": 0.8},
)
bm25_retriever = BM25Retriever.from_documents(chunks)
bm25_retriever.k = 20
small_ensemble = EnsembleRetriever(retrievers=[bm25_retriever, vector_retriever])

# -- Large-chunk retrieval leg --
large_vector_retriever = large_vectorstore.as_retriever(
    search_type='mmr', search_kwargs={"k": 20, "fetch_k": 40,"lambda_mult": 0.8}
)
large_bm25_retriever = BM25Retriever.from_documents(large_chunks)
large_bm25_retriever.k = 20
large_ensemble = EnsembleRetriever(retrievers=[large_bm25_retriever, large_vector_retriever])

# -- Merge + dedup --
dual_retriever = DualEnsembleRetriever(
    small_retriever=small_ensemble,
    large_retriever=large_ensemble,
    dedup_threshold=0.7
)

# -- Cross-encoder reranker --
cross_encoder = HuggingFaceCrossEncoder(model_name="cross-encoder/ms-marco-MiniLM-L-6-v2")
compressor = CrossEncoderReranker(model=cross_encoder, top_n=10)

retriever = ContextualCompressionRetriever(
    base_compressor=compressor,
    base_retriever=dual_retriever
)

print("\n[DEBUG] Loading Prompts & Initializing Memory...")
with open("prompts.yaml", "r") as file:
    config = yaml.safe_load(file)

# Initialize SQLite Memory
session_id = "raunak_test_session_2" 
history = SQLChatMessageHistory(session_id, "sqlite:///chat_memory.db")

# Format the last 4 messages to give the LLM context without blowing up the token limit
past_messages = history.messages[-4:]
chat_history_str = "\n".join([f"{msg.type}: {msg.content}" for msg in past_messages])
if not chat_history_str:
    chat_history_str = "No prior conversation."

# ==========================================
# 2. DYNAMIC QUERY REWRITER
# ==========================================
rewrite_prompt = ChatPromptTemplate.from_messages([
    ("system", config["query_rewriter"]["system"]),
    ("human", config["query_rewriter"]["human"])
])
query_rewriter = rewrite_prompt | llm | StrOutputParser()

# ==========================================
# 3. FINAL GENERATOR 
# ==========================================
qa_prompt = ChatPromptTemplate.from_messages([
    ("system", config["qa_generator"]["system"]),
    ("human", config["qa_generator"]["human"]),
])
question_answer_chain = create_stuff_documents_chain(llm, qa_prompt)

# ==========================================
# 4. SPLIT-BRAIN EXECUTION PIPELINE
# ==========================================
original_user_query = "What context window does the largest Llama 3 model support?"

# A. Rewrite the query (Now with memory context!)
print("\n[DEBUG] Optimizing Query...")
optimized_query = query_rewriter.invoke({
    "input": original_user_query,
    "chat_history": chat_history_str
})
print(f"Original Query:  {original_user_query}")
print(f"Optimized Query: {optimized_query}")

# B. Fetch chunks
print("\n[DEBUG] Fetching Reranked Chunks...")
retrieved_docs = retriever.invoke(optimized_query)

# C. Generate Final Answer
print("\n[DEBUG] Generating Final Answer...")
response = question_answer_chain.invoke({
    "input": original_user_query,
    "context": retrieved_docs
})

# D. Save the interaction to SQLite Memory
history.add_user_message(original_user_query)
history.add_ai_message(response)

# ==========================================
# 5. OUTPUT
# ==========================================
print("\n=== FINAL ANSWER ===")
print(response)

print("\n=== CITATIONS ===")
if retrieved_docs:
    for doc in retrieved_docs[:3]: # Limit to top 3 to keep UI clean
        source_file = os.path.basename(doc.metadata.get('source', 'Unknown'))
        page_num = doc.metadata.get('page', 'Unknown')
        print(f"- {source_file} (Page {page_num})")
else:
    print("- No citations found.")


[DEBUG] Loading Prompts & Initializing Memory...

[DEBUG] Optimizing Query...
Original Query:  What context window does the largest Llama 3 model support?
Optimized Query: Llama 3 model context window size

[DEBUG] Fetching Reranked Chunks...
[DualRetriever] small=33, large=33, after dedup=16

[DEBUG] Generating Final Answer...

=== FINAL ANSWER ===
The largest Llama 3 model supports a context window of up to 128K tokens.

=== CITATIONS ===
- Llama3.pdf (Page 0)
- Llama3.pdf (Page 13)
- Llama3.pdf (Page 23)


In [16]:
import ragas
print(f"--- DEBUG: Currently running Ragas Version {ragas.__version__} ---")

from datasets import Dataset
from ragas import evaluate
from ragas.run_config import RunConfig # <-- IMPORTED THE THROTTLE CONTROLLER

# Import the uppercase Classes, not the lowercase aliases
from ragas.metrics import Faithfulness, AnswerRelevancy

# ==========================================
# 1. THE GOLDEN DATASET (40 Questions)
# ==========================================
test_questions = [
    "What problem does FlashAttention-2 primarily address in Transformer models?",
    "How does FlashAttention-2 improve GPU efficiency compared to FlashAttention?",
    "What throughput percentage of theoretical maximum FLOPs/s can FlashAttention-2 achieve on A100 GPUs?",
    "Why are non-matmul FLOPs important to reduce in FlashAttention-2?",
    "What memory complexity does standard attention require?",
    "What is the main architectural innovation introduced in the Transformer paper?",
    "What is Scaled Dot-Product Attention?",
    "Why does the Transformer architecture allow more parallelization than RNNs?",
    "How many encoder and decoder layers are used in the original Transformer model?",
    "What is the purpose of multi-head attention in Transformers?",
    "What is LoRA in large language model adaptation?",
    "How does LoRA reduce training costs for large language models?",
    "What is one major advantage of LoRA during inference?",
    "Why is full fine-tuning difficult for very large models like GPT-3?",
    "What hypothesis motivates the LoRA approach?",
    "How does LoRA differ from adapter-based fine-tuning methods?",
    "What is the largest model size described in the Llama 3 paper?",
    "What context window does the largest Llama 3 model support?",
    "What capabilities are natively supported by Llama 3 models?",
    "How many multilingual tokens were used to pre-train Llama 3?",
    "What training scale in FLOPs was used for the flagship Llama 3 model?",
    "What are the two major stages in developing Llama 3?",
    "What architecture does Llama 3 primarily use?",
    "What optimization technique does Llama 3 use to improve inference efficiency?",
    "What percentage of the final Llama 3 data mix consists of code tokens?",
    "What percentage of the final Llama 3 data mix consists of multilingual tokens?",
    "Why does Llama 3 remove markdown markers during preprocessing?",
    "What de-duplication strategies are applied during Llama 3 data curation?",
    "What is TorchDynamo in PyTorch 2?",
    "What is TorchInductor in PyTorch 2?",
    "What does torch.compile provide in PyTorch 2?",
    "How does TorchDynamo capture computation graphs?",
    "What languages does TorchInductor generate code for?",
    "What inference speedup does TorchInductor achieve on NVIDIA A100 GPUs?",
    "Why are eager mode frameworks preferred by many researchers?",
    "What limitation do eager mode frameworks have compared to graph mode frameworks?",
    "Why is torch.jit.trace considered unsound?",
    "What challenge does torch.jit.script face when capturing Python programs?",
    "What is the main drawback of standard attention implementations?",
    "Why is FlashAttention considered more memory efficient than standard attention?"
]

ground_truths = [
    "FlashAttention-2 addresses the quadratic runtime and memory bottleneck of attention layers in Transformers, especially for long sequence lengths.",
    "FlashAttention-2 improves GPU efficiency through better work partitioning, reducing non-matmul FLOPs, parallelizing across sequence lengths, and reducing shared memory communication.",
    "FlashAttention-2 can achieve up to 73% of the theoretical maximum FLOPs/s on A100 GPUs.",
    "Non-matmul FLOPs are important to reduce because GPUs are highly optimized for matrix multiplication operations using Tensor Cores.",
    "Standard attention requires O(N^2) memory complexity because it materializes the attention score and probability matrices.",
    "The Transformer architecture relies entirely on attention mechanisms without using recurrence or convolutions.",
    "Scaled Dot-Product Attention computes attention weights using scaled dot products between queries and keys, followed by a softmax operation applied to values.",
    "The Transformer allows more parallelization because it removes sequential recurrence and processes sequence positions in parallel.",
    "The original Transformer model uses 6 encoder layers and 6 decoder layers.",
    "Multi-head attention allows the model to attend to information from different representation subspaces simultaneously.",
    "LoRA, or Low-Rank Adaptation, freezes pretrained model weights and injects trainable low-rank matrices into Transformer layers.",
    "LoRA reduces training costs by drastically reducing the number of trainable parameters and lowering GPU memory requirements.",
    "LoRA introduces no additional inference latency because the low-rank matrices can be merged into pretrained weights.",
    "Full fine-tuning is difficult because each downstream task requires storing and training a separate massive model with billions of parameters.",
    "LoRA is motivated by the hypothesis that weight updates during adaptation have low intrinsic rank.",
    "LoRA differs from adapter methods by introducing no additional inference latency and directly modifying weight updates through low-rank decomposition.",
    "The largest model described in the Llama 3 paper has 405 billion parameters.",
    "The largest Llama 3 model supports a context window of up to 128K tokens.",
    "Llama 3 natively supports multilinguality, coding, reasoning, and tool usage.",
    "Llama 3 was pre-trained on approximately 15 trillion multilingual tokens.",
    "The flagship Llama 3 model was trained using approximately 3.8 × 10^25 FLOPs.",
    "The two major stages in developing Llama 3 are pre-training and post-training.",
    "Llama 3 primarily uses a standard dense Transformer architecture.",
    "Llama 3 uses grouped query attention (GQA) with 8 key-value heads to improve inference efficiency.",
    "Approximately 17% of the final Llama 3 data mix consists of code tokens.",
    "Approximately 8% of the final Llama 3 data mix consists of multilingual tokens.",
    "Llama 3 removes markdown markers because experiments showed markdown harmed performance compared to plain text.",
    "Llama 3 applies URL-level, document-level, and line-level de-duplication during data curation.",
    "TorchDynamo is a Python-level just-in-time compiler introduced in PyTorch 2 for dynamic graph capture.",
    "TorchInductor is the default compiler backend for TorchDynamo in PyTorch 2.",
    "torch.compile provides graph compilation and compiler optimizations for PyTorch programs while preserving eager execution flexibility.",
    "TorchDynamo captures graphs by dynamically modifying Python bytecode before execution and extracting FX graphs.",
    "TorchInductor generates Triton code for GPUs and C++ code for CPUs.",
    "TorchInductor achieves a 2.27× inference geometric mean speedup on NVIDIA A100 GPUs across real-world models.",
    "Researchers prefer eager mode frameworks because they are easier to understand and debug using standard Python tools.",
    "Eager mode frameworks make graph-level compiler optimizations more difficult because execution occurs operator-by-operator.",
    "torch.jit.trace is considered unsound because it specializes execution paths based on example inputs and may produce incorrect behavior for different inputs.",
    "torch.jit.script struggles because it attempts to statically reimplement the full semantics of Python.",
    "The main drawback of standard attention implementations is the quadratic memory and runtime cost caused by materializing attention matrices.",
    "FlashAttention is more memory efficient because it reduces memory usage from quadratic to linear in sequence length."
]

# ==========================================
# 2. GENERATE SYSTEM ANSWERS
# ==========================================
answers = []
contexts = []

print("Running your Split-Brain pipeline to generate test answers...\n")

for i, q in enumerate(test_questions):
    print(f"[{i+1}/{len(test_questions)}] Question: {q}")
    
    # Step 1: Rewrite the query 
    # (We pass an empty history so tests do not poison each other)
    optimized_query = query_rewriter.invoke({
        "input": q,
        "chat_history": "No prior conversation."
    })
    
    # Step 2: Fetch the chunks using the optimized query
    retrieved_docs = retriever.invoke(optimized_query)
    
    # Step 3: Generate the final answer using the original query and retrieved docs
    response_text = question_answer_chain.invoke({
        "input": q,
        "context": retrieved_docs
    })
    
    # Step 4: Save the outputs for Ragas
    answers.append(response_text)
    
    # Extract just the raw text from the LangChain Document objects
    chunk_texts = [doc.page_content for doc in retrieved_docs]
    contexts.append(chunk_texts)

print("\nGeneration complete! Formatting dataset for Ragas...")

data = {
    "question": test_questions,
    "answer": answers,
    "contexts": contexts,
    "ground_truth": ground_truths
}
dataset = Dataset.from_dict(data)

# ==========================================
# 3. THE AUTOMATED JUDGE
# ==========================================
print("\nGrading the answers using Ragas (Throttled for Free Tier API)...")

eval_metrics = [
    Faithfulness(), 
    AnswerRelevancy(strictness=1)
]

# <-- ADDED RUN_CONFIG TO PREVENT TIMEOUT ERRORS
result = evaluate(
    dataset=dataset,
    metrics=eval_metrics,
    llm=llm, 
    embeddings=embedding_function, 
    raise_exceptions=False,
    run_config=RunConfig(max_workers=1, max_retries=3) 
)

print("\n--- FINAL EVALUATION SCORES ---")
print(result)

df = result.to_pandas()
display(df[['user_input', 'faithfulness', 'answer_relevancy']])

--- DEBUG: Currently running Ragas Version 0.4.3 ---
Running your Split-Brain pipeline to generate test answers...

[1/40] Question: What problem does FlashAttention-2 primarily address in Transformer models?


C:\Users\nikhi\AppData\Local\Temp\ipykernel_11800\2639589066.py:9: DeprecationWarning: Importing Faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import Faithfulness
  from ragas.metrics import Faithfulness, AnswerRelevancy
C:\Users\nikhi\AppData\Local\Temp\ipykernel_11800\2639589066.py:9: DeprecationWarning: Importing AnswerRelevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import AnswerRelevancy
  from ragas.metrics import Faithfulness, AnswerRelevancy


[DualRetriever] small=38, large=36, after dedup=28
[2/40] Question: How does FlashAttention-2 improve GPU efficiency compared to FlashAttention?
[DualRetriever] small=32, large=29, after dedup=13
[3/40] Question: What throughput percentage of theoretical maximum FLOPs/s can FlashAttention-2 achieve on A100 GPUs?
[DualRetriever] small=25, large=28, after dedup=17
[4/40] Question: Why are non-matmul FLOPs important to reduce in FlashAttention-2?
[DualRetriever] small=33, large=31, after dedup=17
[5/40] Question: What memory complexity does standard attention require?
[DualRetriever] small=37, large=33, after dedup=25
[6/40] Question: What is the main architectural innovation introduced in the Transformer paper?
[DualRetriever] small=40, large=38, after dedup=38
[7/40] Question: What is Scaled Dot-Product Attention?
[DualRetriever] small=38, large=36, after dedup=28
[8/40] Question: Why does the Transformer architecture allow more parallelization than RNNs?
[DualRetriever] small=36, large

Evaluating:   0%|          | 0/80 [00:00<?, ?it/s]

Exception raised in Job[42]: OutputParserException(Invalid json output: The output string did not satisfy the constraints given in the prompt. The output string should be in JSON format and comply with the specified schema. Here is the corrected output string in JSON format:

{
    "statements": [
        "The development of Llama 3 language models is a process.",
        "The process of developing Llama 3 language models comprises two main stages."
    ]
}
For troubleshooting, visit: https://docs.langchain.com/oss/python/langchain/errors/OUTPUT_PARSING_FAILURE )
Exception raised in Job[64]: LLMDidNotFinishException(The LLM generation was not completed. Please increase the max_tokens and try again.)



--- FINAL EVALUATION SCORES ---
{'faithfulness': 0.9412, 'answer_relevancy': 0.9135}


,user_input,faithfulness,answer_relevancy
0,What problem does FlashAttention-2 primarily a...,1.000000,0.922475
1,How does FlashAttention-2 improve GPU efficien...,1.000000,1.000000
2,What throughput percentage of theoretical maxi...,1.000000,0.975208
3,Why are non-matmul FLOPs important to reduce i...,1.000000,1.000000
4,What memory complexity does standard attention...,1.000000,0.782545
5,What is the main architectural innovation intr...,1.000000,1.000000
6,What is Scaled Dot-Product Attention?,1.000000,0.735956
7,Why does the Transformer architecture allow mo...,1.000000,0.919161
8,How many encoder and decoder layers are used i...,1.000000,0.968989
9,What is the purpose of multi-head attention in...,0.750000,0.954121


In [17]:
display(df.iloc[12,:])

user_input            What is one major advantage of LoRA during inf...
retrieved_contexts    [LoRA can reduce the number of trainable param...
response              LoRA introduces no additional inference latenc...
reference             LoRA introduces no additional inference latenc...
faithfulness                                                        1.0
answer_relevancy                                               0.804818
Name: 12, dtype: object